In [ ]:
import io
import json
import zipfile
from pathlib import Path
from typing import Literal, Any
from urllib.parse import quote
from collections.abc import Callable

import httpx
from tqdm.auto import tqdm
from minsearch import AppendableIndex
from pydantic import BaseModel, ValidationError
from anthropic import Anthropic
from anthropic.types import ToolUseBlock
from gitsource import chunk_documents

In [81]:
client = Anthropic()


PROMPT_TEMPLATE = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

INSTRUCTIONS = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search
2) in the second interation, analyze the results from the previous search
   and perform 2 more searches
3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Pulumi, so you don't need to 
include the word 'pulumi' in search results
"""

search_tool = {
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index",
            }
        },
        "required": ["query"],
    },
}

add_entry_tool = {
    "name": "add_entry",
    "description": "Add a new documentation entry to the index.",
    "input_schema": {
        "type": "object",
        "properties": {
            "filename": {
                "type": "string",
                "description": "The source filename associated with the entry",
            },
            "content": {
                "type": "string",
                "description": "The full content of the documentation entry",
            },
        },
        "required": ["filename", "content"],
    },
}

In [82]:
class RAGResponse(BaseModel):
    answer: str
    found_answer: bool
    confidence: float
    confidence_explanation: str
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"]
    followup_questions: list[str]

In [93]:
class AgenticRAG:
    MAX_OUTPUT_TOKENS = 32000

    def __init__(
        self,
        index: AppendableIndex,
        llm_client: Anthropic,
        output_type: type[BaseModel],
        tools: list[dict],
        instructions: str = INSTRUCTIONS,
    ):
        self.index = index
        self.llm_client = llm_client
        self.output_type = output_type
        self.tools = tools
        self.instructions = instructions

    def _find_tool(self, tool_name: str) -> Callable[..., Any]:
        _tool_functions: dict[str, Callable[..., Any]] = {
            "search": self.search,
            "add_entry": self.add_entry,
        }
        tool_func = _tool_functions.get(tool_name)
        if tool_func is None:
            raise ValueError(f"{tool_name} tool does not exist")
        return tool_func

    def _invoke_tool(
        self,
        tool_id: str,
        tool_name: str,
        args: dict[str, Any],
    ) -> dict[str, Any]:
        try:
            tool_func = self._find_tool(tool_name=tool_name)
            print(f"\nAgent found {tool_func.__name__} tool function")
            print(f"Agent calling {tool_func.__name__} tool with {json.dumps(args)}")
            tool_result = tool_func(**args)
            print(f"Agent called {tool_name} tool")
        except ValueError as e:
            print(f"Agent did not find {tool_name} tool function. See error: {e}")
            tool_result = 'not found tool "{tool_name}"'

        return {
            "type": "tool_result",
            "tool_use_id": tool_id,
            "content": json.dumps(tool_result, indent=2),
        }

    def _loop_tool_calls(self, tool_calls: list[ToolUseBlock]):
        tool_results = []
        for tool_use in tool_calls:
            tool_call_id = tool_use.id
            tool_call_arguments = tool_use.input
            invoked_tool_result = self._invoke_tool(
                tool_id=tool_call_id,
                tool_name=tool_use.name,
                args=tool_call_arguments
            )
            tool_results.append(invoked_tool_result)
        return tool_results

    def add_entry(self, filename: str, content: str):
        entry = {
            "start": 0,
            "content": content,
            "filename": filename,
        }
        self.index.append(entry)
        return "Entry added"

    def search(self, query: str, top_k: int = 5):
        return self.index.search(query, num_results=top_k)

    def build_prompt(self, question: str, search_results: list):
        context = json.dumps(search_results, indent=2)
        return PROMPT_TEMPLATE.format(question=question, context=context)

    def llm(
        self,
        max_tokens: int,
        model: str = "claude-opus-5",
        messages: list | None = None,
    ):
        if not messages:
            messages = []

        response = self.llm_client.messages.parse(
            model=model,
            max_tokens=max_tokens,
            system=INSTRUCTIONS,
            messages=messages,  # type: ignore
            tools=self.tools,  # type: ignore
            output_format=self.output_type,
        )
        return response

    def dynamic_max_tokens(
        self,
        base_max_tokens: int,
        messages: list[dict[str, Any]],
        tokens_per_message: int = 512,
        cap: int = MAX_OUTPUT_TOKENS,
    ) -> int:
        """Grow the output budget as the conversation history grows.

        Starts from `base_max_tokens` and adds `tokens_per_message` for every
        message already in the history, so longer conversations get more room
        to answer. Capped at `cap` to stay within the model's output limit.
        """
        return min(base_max_tokens + tokens_per_message * len(messages), cap)

    def loop(
        self,
        max_tokens: int,
        messages: list[dict[str, Any]],
        max_rounds: int = 5,
        model: str = "claude-opus-5",
        max_retries: int = 2,
    ):
        for _ in range(max_rounds):
            # Scale the output budget with the size of the history so far.
            recalculated_max_tokens = self.dynamic_max_tokens(
                base_max_tokens=max_tokens,
                messages=messages,
            )

            # A long answer can run out of output tokens before the JSON closes,
            # which leaves an unparseable structured response. Retry with more room.
            response = None
            for _ in range(max_retries + 1):
                try:
                    response = self.llm(
                        model=model,
                        max_tokens=recalculated_max_tokens,
                        messages=messages,  # type: ignore
                    )
                    break
                except ValidationError:
                    next_tokens = min(
                        recalculated_max_tokens * 2, self.MAX_OUTPUT_TOKENS
                    )
                    if next_tokens == recalculated_max_tokens:
                        raise
                    print(
                        f"Structured output was cut off at "
                        f"{recalculated_max_tokens} tokens; retrying with {next_tokens}"
                    )
                    recalculated_max_tokens = next_tokens

            if response is None:
                raise RuntimeError("No valid structured response after retries")

            tool_calls = [block for block in response.content if block.type == "tool_use"]
            if not tool_calls:
                # Record the final assistant turn so the history keeps
                # alternating (user -> assistant -> user -> ...) and stays reusable.
                messages.append(
                    {
                        "role": "assistant",
                        "content": "".join(
                            block.text
                            for block in response.content
                            if block.type == "text"
                        ),
                    }
                )
                return response, messages

            # Each tool round is two turns: assistant tool_use, then user tool_result.
            # `extend` keeps `messages` flat -- never nest a list of messages.
            tool_results = self._loop_tool_calls(tool_calls=tool_calls)
            messages.extend(
                [
                    {
                        "role": "assistant",
                        "content": [
                            tool_call.model_dump(exclude={"toolset_name", "caller"})
                            for tool_call in tool_calls
                        ],
                    },
                    {"role": "user", "content": tool_results},
                ]
            )

        raise RuntimeError(f"No final answer after {max_rounds} rounds")

    def loop_invoke(
        self,
        max_tokens: int = 2056,
        messages: list[dict[str, Any]] | None = None,
        search: bool = True,
    ):
        messages = [] if not messages else messages
        while True:
            # Prompt user for query
            query = input("> You: ")
            if query.strip().lower() == "stop":
                break

            # Only run fresh retrieval for the first turn; follow-ups reuse the
            # knowledge already in the history.
            if search:
                search_results = self.search(query=query)
                user_prompt = self.build_prompt(question=query, search_results=search_results)
            else:
                user_prompt = query

            # Continue an existing conversation when history is passed in,
            # otherwise start a new one. Either way, append the new user turn.
            messages.append({"role": "user", "content": user_prompt})

            final_response, messages = self.loop(
                max_tokens=max_tokens,
                messages=messages,
            )
            structured_response = final_response.parsed_output

            print(f"\nTokens: \n- Inputs: {final_response.usage.input_tokens}\n- Outputs: {final_response.usage.output_tokens}")
            print(f"\nAnswer: {structured_response.answer}")  # type: ignore
            print(f"Found answer: {structured_response.found_answer}")  # type: ignore
            print(f"Confidence: {structured_response.confidence}")  # type: ignore
            print(f"Answer type: {structured_response.answer_type}")  # type: ignore
            print(f"Follow-up questions: {structured_response.followup_questions}")  # type: ignore
            print("================================================================================")

In [84]:
def get_or_download_documentation(
    repo_owner: str,
    repo_name: str,
    branch_name: str,
    docs_path: str = "docs",
    save_dir: str | None = None,
) -> bytes:
    """
    Return the Markdown documentation as a zip archive, reusing a local copy if one already exists, and downloading it otherwise.

    Args:
        repo_owner (str): The owner of the GitHub repository.
        repo_name (str): The name of the GitHub repository.
        branch_name (str): The branch to download from.
        docs_path (str): Path of the documentation folder inside the repository.
        save_dir (str | None): Local directory to reuse/save the files in. If
            None, the files are only returned as an in-memory zip.

    Returns:
        bytes: A zip archive containing only the Markdown files under `docs_path`.
    """

    def _get(doc_dir: str | None = None) -> bytes | None:
        """Build a zip from an already-downloaded local directory.

        Args:
            doc_dir (str | None): Directory that may already contain the files.

        Returns:
            bytes | None: A zip archive of the local files, or None if the
            directory is missing or empty.
        """
        if doc_dir is None:
            return None

        out_dir = Path(doc_dir)
        existing_files = sorted(out_dir.rglob("*")) if out_dir.exists() else []
        existing_files = [f for f in existing_files if f.is_file()]
        if not existing_files:
            return None

        print(f"Using already-downloaded documentation in '{doc_dir}' ({len(existing_files)} files)")

        buffer = io.BytesIO()
        with zipfile.ZipFile(buffer, "w") as archive:
            for file in existing_files:
                archive.write(file, file.relative_to(out_dir).as_posix())
        return buffer.getvalue()

    def _download() -> bytes:
        """Download only the Markdown documentation from GitHub.

        Returns:
            bytes: A zip archive containing only the Markdown files under
            `docs_path`.
        """
        api = "https://api.github.com"
        raw = "https://raw.githubusercontent.com"

        with httpx.Client(follow_redirects=True) as client:
            # Resolve the branch to its root tree SHA
            branch = client.get(f"{api}/repos/{repo_owner}/{repo_name}/branches/{branch_name}")
            branch.raise_for_status()
            root_tree_sha = branch.json()["commit"]["commit"]["tree"]["sha"]
            print(f"Retrieved root tree SHA: {root_tree_sha}")

            # Locate the docs subtree inside the root tree
            root_tree = client.get(f"{api}/repos/{repo_owner}/{repo_name}/git/trees/{root_tree_sha}")
            root_tree.raise_for_status()
            docs_entry = next(
                (item for item in root_tree.json()["tree"] if item["path"] == docs_path),
                None,
            )
            if docs_entry is None:
                raise ValueError(f"Path '{docs_path}' not found in {repo_owner}/{repo_name}@{branch_name}")
            print(f"Retrieved docs subtree '{docs_path}' (SHA: {docs_entry['sha']})")

            # Recursively list every file under docs/
            docs_tree = client.get(
                f"{api}/repos/{repo_owner}/{repo_name}/git/trees/{docs_entry['sha']}?recursive=1"
            )
            docs_tree.raise_for_status()
            print(f"Retrieved recursive tree for '{docs_path}'")

            # Collect the Markdown files to download
            markdown_files = [
                item["path"]
                for item in docs_tree.json().get("tree", [])
                if item.get("type") == "blob" and item["path"].endswith((".md", ".mdx"))
            ]
            print(f"Found {len(markdown_files)} Markdown files to download")

            # Download each Markdown file and package it into an in-memory zip
            buffer = io.BytesIO()
            with zipfile.ZipFile(buffer, "w") as archive:
                for path in tqdm(markdown_files, desc="Downloading docs", unit="file"):
                    file_url = f"{raw}/{repo_owner}/{repo_name}/{branch_name}/{quote(f'{docs_path}/{path}')}"
                    response = client.get(file_url)
                    response.raise_for_status()
                    archive.writestr(path, response.content)

        return buffer.getvalue()

    # Reuse the local copy if it already exists
    data = _get(save_dir)
    if data is not None:
        return data

    # Otherwise download and (optionally) save to disk
    data = _download()

    if save_dir is not None:
        out_dir = Path(save_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(io.BytesIO(data)) as zf:
            num_files = len(zf.namelist())
            zf.extractall(out_dir)
        print(f"Extracted {num_files} files to '{save_dir}'")

    return data

In [91]:
def retrieve_documentation(
    repo_owner: str,
    repo_name: str,
    branch_name: str,
    docs_path: str = "docs",
    save_dir: str | None = None,
):
    response_contents = get_or_download_documentation(
        repo_owner=repo_owner,
        repo_name=repo_name,
        branch_name=branch_name,
        docs_path=docs_path,
        save_dir=save_dir,
    )
    documents = []
    with zipfile.ZipFile(io.BytesIO(response_contents)) as zip_ref:
        for file_path in zip_ref.namelist():
            with zip_ref.open(file_path) as file:
                content = file.read().decode('utf-8')
                doc = {
                    'content': content,
                    'filename': file_path
                }
                documents.append(doc)
    return documents

In [92]:
documents = retrieve_documentation(
    repo_owner="pulumi",
    repo_name="docs",
    branch_name="master",
    docs_path="content",
    save_dir="docs",
)
documents[:5]

Using already-downloaded documentation in 'docs' (2186 files)


[{'content': '---\ntitle: The infrastructure as code platform for the AI era – Pulumi\nmeta_desc: Pulumi is an open-source infrastructure as code platform that helps humans and agents build and manage cloud infrastructure with real programming languages.\nmeta_image: /images/home/home-meta.png\n\ninclude_organization_schema: true\n\nsections:\n  - type: hero\n    layout: split\n    cta_secondary_text: Download open source\n    cta_secondary_link: /docs/install/\n    badge_highlight_text: "New:"\n    badge_text: "Native support for Terraform and HCL"\n    badge_link: /releases/terraform-state-backend-modules-hcl/\n    title: "*Unleash agents* <br>on your infrastructure."\n    description: |\n      Ship infrastructure faster with tools that let agents do what they do best — without letting them go off the rails.\n    anchor: hero\n    hero_animation: agent-loop\n\n  - type: logo_grid\n    title: Trusted by over 4,000 innovative companies\n    logos:\n      - snowflake\n      - nvidia\n  

In [87]:
document_chunks = chunk_documents(documents, size=3000, step=1500)

In [88]:
index = AppendableIndex(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(document_chunks)

In [89]:
rag = AgenticRAG(
    index=index,
    llm_client=client,
    output_type=RAGResponse,
    tools=[search_tool, add_entry_tool],
)

In [ ]:
question = "How do I launch an EC2 in a private net in AWS?"
rag.loop_invoke(max_tokens=8192)


Agent found search tool function
Agent calling search tool with {"query": "launch EC2 instance in private subnet VPC"}
Agent called search tool

Agent found search tool function
Agent calling search tool with {"query": "aws.ec2.Instance resource example subnetId ami instanceType"}
Agent called search tool

Agent found search tool function
Agent calling search tool with {"query": "awsx ec2 Vpc private subnets NAT gateway component"}
Agent called search tool

Tokens: 
- Inputs: 32398
- Outputs: 2884

Answer: Yes — here's how to do it with Pulumi, based on the docs.

**Search rationale (iteration 1):** I first searched "launch EC2 instance in private subnet VPC" to find any end‑to‑end guidance combining networking and compute. That returned VPC/private‑subnet examples (Redshift, import docs, AWSX state migration) but no complete EC2‑in‑private‑subnet walkthrough, so in iteration 2 I ran two targeted searches: one on the `aws.ec2.Instance` resource arguments (`ami`, `instanceType`, `subne